# SemanticEmbed: The Six Dimensions

A deep dive into what each of the six structural measurements means, why they matter, and how to read them.

[GitHub](https://github.com/jmurray10/semanticembed-sdk)


In [ ]:
!pip install -q semanticembed
from semanticembed import encode, report
print("Ready.")


---
## The Encoding at a Glance

Every node in a directed graph gets six numbers. Together they form a coordinate vector in 6-dimensional Euclidean space that fully describes the node's structural role.

| # | Dimension | What It Measures | Range |
|---|-----------|-----------------|-------|
| 1 | **Depth** | Pipeline position (entry to sink) | 0.0 -- 1.0 |
| 2 | **Independence** | Lateral redundancy at same stage | 0.0 -- 1.0 |
| 3 | **Hierarchy** | Module / group membership | 0.0 -- 1.0 |
| 4 | **Throughput** | Share of total traffic | 0.0 -- 1.0 |
| 5 | **Criticality** | Share of end-to-end paths | 0.0 -- 1.0 |
| 6 | **Fanout** | Broadcaster (1.0) vs aggregator (0.0) | 0.0 -- 1.0 |

These are mathematically independent. Knowing any five tells you nothing about the sixth.


---
## Dimension 1: Depth

**What:** Where a node sits in the execution pipeline -- from entry point (0.0) to deepest sink (1.0).

**Why it matters:** Depth tells you the blast radius direction. A failure at depth 0.0 cascades forward through the entire graph. A failure at depth 1.0 affects nothing downstream.


In [ ]:
# A simple pipeline -- depth increases left to right
pipeline = [
    ("ingestion", "validation"),
    ("validation", "transform"),
    ("transform", "enrichment"),
    ("enrichment", "storage"),
]

result = encode(pipeline)
print("LINEAR PIPELINE -- depth increases with each hop")
print("=" * 60)
for node, v in sorted(result.vectors.items(), key=lambda x: x[1][0]):
    print(f"  {node:<15} depth={v[0]:.3f}")


---
## Dimension 2: Independence

**What:** How many other nodes exist at the same depth level. Measures lateral redundancy.

**Why it matters:** Independence = 0.0 means this node is the ONLY node at its depth. There is no parallel alternative. It is a structural chokepoint regardless of whether it is healthy today.

**This dimension has no equivalent in standard graph centrality.** Maximum correlation with all 10 standard centrality measures across real applications: |r| = 0.369.


In [ ]:
# Wide fan-out -- services at depth 1 have high independence
wide = [
    ("gateway", "service-a"),
    ("gateway", "service-b"),
    ("gateway", "service-c"),
    ("gateway", "service-d"),
    ("service-a", "db"),
    ("service-b", "db"),
    ("service-c", "db"),
    ("service-d", "db"),
]

result = encode(wide)
print("WIDE ARCHITECTURE -- high independence at depth 1")
print("=" * 60)
for node, v in sorted(result.vectors.items(), key=lambda x: x[1][0]):
    print(f"  {node:<15} depth={v[0]:.3f}  independence={v[1]:.3f}")


---
## Dimension 3: Hierarchy

**What:** Which module or logical group a node belongs to.

**Why it matters:** Hierarchy captures community structure. Nodes in the same module have similar hierarchy values. Useful for detecting cross-module dependencies and module boundary violations.


In [ ]:
# Two separate modules feeding into a shared database
two_modules = [
    ("orders-api",    "orders-processor"),
    ("orders-processor", "orders-db"),
    ("users-api",     "users-processor"),
    ("users-processor", "users-db"),
    ("orders-processor", "shared-cache"),
    ("users-processor",  "shared-cache"),
]

result = encode(two_modules)
print("TWO-MODULE ARCHITECTURE")
print("=" * 60)
for node, v in sorted(result.vectors.items(), key=lambda x: x[1][2]):
    print(f"  {node:<20} hierarchy={v[2]:.3f}")


---
## Dimension 4: Throughput

**What:** What fraction of total traffic flows through this node.

**Why it matters:** High throughput + low independence = hidden bottleneck. The node carries a disproportionate share of traffic with no backup.

Independent of depth: a deep node can have high or low throughput.


In [ ]:
# Hub-and-spoke -- the hub has high throughput
hub_spoke = [
    ("client-1", "api-hub"),
    ("client-2", "api-hub"),
    ("client-3", "api-hub"),
    ("api-hub",  "backend-a"),
    ("api-hub",  "backend-b"),
    ("api-hub",  "backend-c"),
    ("backend-a", "database"),
    ("backend-b", "database"),
    ("backend-c", "database"),
]

result = encode(hub_spoke)
print("HUB-AND-SPOKE -- hub carries most traffic")
print("=" * 60)
for node, v in sorted(result.vectors.items(), key=lambda x: -x[1][3]):
    print(f"  {node:<15} throughput={v[3]:.3f}  independence={v[1]:.3f}")


---
## Dimension 5: Criticality

**What:** How many end-to-end paths through the graph depend on this node.

**Why it matters:** A node with criticality 0.5 sits on half of all paths. If it fails, half of all end-to-end flows break. This is independent of throughput -- a low-traffic node can be highly critical if it bridges two large subgraphs.


In [ ]:
# Diamond shape -- the middle nodes are critical bridges
diamond = [
    ("source",  "left-branch"),
    ("source",  "right-branch"),
    ("left-branch",  "merge-point"),
    ("right-branch", "merge-point"),
    ("merge-point", "sink-a"),
    ("merge-point", "sink-b"),
    ("merge-point", "sink-c"),
]

result = encode(diamond)
print("DIAMOND -- merge-point bridges source to all sinks")
print("=" * 60)
for node, v in sorted(result.vectors.items(), key=lambda x: -x[1][4]):
    print(f"  {node:<15} criticality={v[4]:.3f}  fanout={v[5]:.3f}")


---
## Dimension 6: Fanout

**What:** Whether a node amplifies (broadcasts to many) or aggregates (collects from many). Out-degree / total degree.

**Why it matters:** High fanout = amplification risk. If this node fails or slows, the impact multiplies across all downstream dependents. Low fanout = convergence sink. Many services depend on this single aggregation point.


In [ ]:
# Contrasting broadcaster vs aggregator
fanout_example = [
    ("broadcaster", "target-1"),
    ("broadcaster", "target-2"),
    ("broadcaster", "target-3"),
    ("broadcaster", "target-4"),
    ("broadcaster", "target-5"),
    ("source-1", "aggregator"),
    ("source-2", "aggregator"),
    ("source-3", "aggregator"),
    ("source-4", "aggregator"),
    ("source-5", "aggregator"),
]

result = encode(fanout_example)
print("BROADCASTER vs AGGREGATOR")
print("=" * 60)
for node in ["broadcaster", "aggregator"]:
    v = result.vectors[node]
    label = "broadcasts to 5" if v[5] > 0.5 else "collects from 5"
    print(f"  {node:<15} fanout={v[5]:.3f}  ({label})")


---
## Structural Risk Patterns

Combinations of dimension values reveal risk:

| Pattern | Dimensions | Risk |
|---------|-----------|------|
| **Amplification risk** | High fanout + high criticality | Failure cascades to many services across many paths |
| **Convergence sink** | Low independence + low fanout | Many services depend on one aggregation point |
| **Structural SPOF** | Low independence + high criticality | Only node at its depth, on most paths |
| **Hidden bottleneck** | High throughput + low independence | Carries most traffic with no redundancy |

The risk report detects these automatically. See [01 - Quickstart](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/01_quickstart.ipynb) for the full walkthrough.

---

*Patent pending. Application #63/994,075.*


## Cache repeat encodes during exploration

When you're iterating on the same edge list (tweaking visualizations, building
slides), set ``cache=True`` to skip the HTTP round trip on repeat calls:

```python
result = se.encode(edges, cache=True)   # 1st call: HTTP
result = se.encode(edges, cache=True)   # 2nd call: 0ms, no HTTP
se.clear_encode_cache()                 # explicit eviction if needed
```

The cache key is order-independent, so reordering the same edges still hits.
